In [2]:
from pathlib import Path
import os

REPO_DIR = Path(r"C:\Users\19734\Desktop\DAVIS\winter quarter 2026\EEC 174AY\Senior Design\Eco-Cars-Depth-Estimation-2026")
DA_V2_DIR = REPO_DIR / "Depth-Anything-V2"
SEGMENT_DIR = Path(r"C:\Users\19734\Desktop\DAVIS\winter quarter 2026\EEC 174AY\Senior Design\Data&Dataset\processed_data-20260324T234031Z-3-006\processed_data\individual_files_training_segment-10082223140073588526_6140_000_6160_000_with_camera_labels")


OUTPUT_DIR = REPO_DIR / "data" / "depth_anything_v2"
RAW_PNG_DIR = OUTPUT_DIR / "raw_png"
RAW_NPY_DIR = OUTPUT_DIR / "raw_npy"
PLOTS_DIR = OUTPUT_DIR / "plots"

for p in [OUTPUT_DIR, RAW_PNG_DIR, RAW_NPY_DIR, PLOTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

IMAGES_DIR = SEGMENT_DIR / "images"
GT_STACK_PATH = SEGMENT_DIR / "depth_stack.npy"
GT_STEMS_PATH = SEGMENT_DIR / "depth_stack_stems.npy"

print("REPO_DIR      =", REPO_DIR)
print("DA_V2_DIR     =", DA_V2_DIR)
print("SEGMENT_DIR   =", SEGMENT_DIR)
print("IMAGES_DIR    =", IMAGES_DIR)
print("GT_STACK_PATH =", GT_STACK_PATH)
print("GT_STEMS_PATH =", GT_STEMS_PATH)
print("OUTPUT_DIR    =", OUTPUT_DIR)

REPO_DIR      = C:\Users\19734\Desktop\DAVIS\winter quarter 2026\EEC 174AY\Senior Design\Eco-Cars-Depth-Estimation-2026
DA_V2_DIR     = C:\Users\19734\Desktop\DAVIS\winter quarter 2026\EEC 174AY\Senior Design\Eco-Cars-Depth-Estimation-2026\Depth-Anything-V2
SEGMENT_DIR   = C:\Users\19734\Desktop\DAVIS\winter quarter 2026\EEC 174AY\Senior Design\Data&Dataset\processed_data-20260324T234031Z-3-006\processed_data\individual_files_training_segment-10082223140073588526_6140_000_6160_000_with_camera_labels
IMAGES_DIR    = C:\Users\19734\Desktop\DAVIS\winter quarter 2026\EEC 174AY\Senior Design\Data&Dataset\processed_data-20260324T234031Z-3-006\processed_data\individual_files_training_segment-10082223140073588526_6140_000_6160_000_with_camera_labels\images
GT_STACK_PATH = C:\Users\19734\Desktop\DAVIS\winter quarter 2026\EEC 174AY\Senior Design\Data&Dataset\processed_data-20260324T234031Z-3-006\processed_data\individual_files_training_segment-10082223140073588526_6140_000_6160_000_with_camera_l

In [3]:
print("IMAGES_DIR exists:", IMAGES_DIR.exists())
print("GT_STACK_PATH exists:", GT_STACK_PATH.exists())
print("GT_STEMS_PATH exists:", GT_STEMS_PATH.exists())

if IMAGES_DIR.exists():
    image_files = list(IMAGES_DIR.glob("*"))
    print("Number of files in images/:", len(image_files))
    print("First 5 image files:", [p.name for p in image_files[:5]])

IMAGES_DIR exists: True
GT_STACK_PATH exists: True
GT_STEMS_PATH exists: True
Number of files in images/: 197
First 5 image files: ['frame_00000.png', 'frame_00001.png', 'frame_00002.png', 'frame_00003.png', 'frame_00004.png']


In [4]:
from pathlib import Path
import urllib.request

CHECKPOINTS_DIR = DA_V2_DIR / "checkpoints"
CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)

CKPT_PATH = CHECKPOINTS_DIR / "depth_anything_v2_vitb.pth"
CKPT_URL = "https://huggingface.co/depth-anything/Depth-Anything-V2-Base/resolve/main/depth_anything_v2_vitb.pth"

if not CKPT_PATH.exists():
    print("Downloading checkpoint...")
    urllib.request.urlretrieve(CKPT_URL, CKPT_PATH)
    print("Downloaded to:", CKPT_PATH)
else:
    print("Checkpoint already exists:", CKPT_PATH)

Checkpoint already exists: C:\Users\19734\Desktop\DAVIS\winter quarter 2026\EEC 174AY\Senior Design\Eco-Cars-Depth-Estimation-2026\Depth-Anything-V2\checkpoints\depth_anything_v2_vitb.pth


In [5]:
import numpy as np
from pathlib import Path

image_files = sorted([p for p in IMAGES_DIR.iterdir() if p.suffix.lower() in [".png", ".jpg", ".jpeg"]])

assert len(image_files) > 0, f"No images found in {IMAGES_DIR}"
assert GT_STACK_PATH.exists(), f"Missing {GT_STACK_PATH}"
assert GT_STEMS_PATH.exists(), f"Missing {GT_STEMS_PATH}"

gt_stack = np.load(GT_STACK_PATH, allow_pickle=True)
gt_stems = np.load(GT_STEMS_PATH, allow_pickle=True)

print("num images:", len(image_files))
print("gt_stack shape:", gt_stack.shape)
print("gt_stems shape:", gt_stems.shape)
print("first 10 image stems:", [p.stem for p in image_files[:10]])
print("first 10 gt stems:", gt_stems[:10])

num images: 197
gt_stack shape: (197, 1280, 1920)
gt_stems shape: (197,)
first 10 image stems: ['frame_00000', 'frame_00001', 'frame_00002', 'frame_00003', 'frame_00004', 'frame_00005', 'frame_00006', 'frame_00007', 'frame_00008', 'frame_00009']
first 10 gt stems: ['frame_00000' 'frame_00001' 'frame_00002' 'frame_00003' 'frame_00004'
 'frame_00005' 'frame_00006' 'frame_00007' 'frame_00008' 'frame_00009']


In [6]:
from pathlib import Path

run_py = DA_V2_DIR / "run.py"
ckpt_dir = DA_V2_DIR / "checkpoints"

print("DA_V2_DIR exists:", DA_V2_DIR.exists())
print("run.py exists:", run_py.exists())
print("checkpoints dir exists:", ckpt_dir.exists())

if ckpt_dir.exists():
    print("checkpoint files:")
    for p in ckpt_dir.iterdir():
        print(" -", p.name)

DA_V2_DIR exists: True
run.py exists: True
checkpoints dir exists: True
checkpoint files:
 - depth_anything_v2_vitb.pth


In [7]:
import subprocess
import sys
from pathlib import Path

run_py = DA_V2_DIR / "run.py"
assert run_py.exists(), f"Cannot find run.py at {run_py}"

cmd = [
    sys.executable, str(run_py),
    "--encoder", "vitb",
    "--img-path", str(IMAGES_DIR),
    "--outdir", str(RAW_PNG_DIR),
    "--pred-only",
    "--grayscale"
]

print("Running inference...")
print(" ".join(cmd))

result = subprocess.run(
    cmd,
    cwd=str(DA_V2_DIR),
    capture_output=True,
    text=True
)

print("return code:", result.returncode)
print("STDOUT:\n", result.stdout)
print("STDERR:\n", result.stderr)

if result.returncode != 0:
    raise RuntimeError("Inference failed.")
else:
    print("Inference complete.")

Running inference...
c:\Users\19734\Desktop\DAVIS\winter quarter 2026\EEC 174AY\Senior Design\Eco-Cars-Depth-Estimation-2026\.venv\Scripts\python.exe C:\Users\19734\Desktop\DAVIS\winter quarter 2026\EEC 174AY\Senior Design\Eco-Cars-Depth-Estimation-2026\Depth-Anything-V2\run.py --encoder vitb --img-path C:\Users\19734\Desktop\DAVIS\winter quarter 2026\EEC 174AY\Senior Design\Data&Dataset\processed_data-20260324T234031Z-3-006\processed_data\individual_files_training_segment-10082223140073588526_6140_000_6160_000_with_camera_labels\images --outdir C:\Users\19734\Desktop\DAVIS\winter quarter 2026\EEC 174AY\Senior Design\Eco-Cars-Depth-Estimation-2026\data\depth_anything_v2\raw_png --pred-only --grayscale
return code: 0
STDOUT:
 Progress 1/197: C:\Users\19734\Desktop\DAVIS\winter quarter 2026\EEC 174AY\Senior Design\Data&Dataset\processed_data-20260324T234031Z-3-006\processed_data\individual_files_training_segment-10082223140073588526_6140_000_6160_000_with_camera_labels\images\frame_00000

In [8]:
pred_pngs = sorted(RAW_PNG_DIR.glob("*.png"))
print("num predicted pngs:", len(pred_pngs))
print("first 10 predicted files:", [p.name for p in pred_pngs[:10]])

num predicted pngs: 197
first 10 predicted files: ['frame_00000.png', 'frame_00001.png', 'frame_00002.png', 'frame_00003.png', 'frame_00004.png', 'frame_00005.png', 'frame_00006.png', 'frame_00007.png', 'frame_00008.png', 'frame_00009.png']


In [9]:
import cv2
import numpy as np
from pathlib import Path

depth_png_files = sorted([p for p in RAW_PNG_DIR.iterdir() if p.suffix.lower() == ".png"])
assert len(depth_png_files) > 0, f"No depth PNG files found in {RAW_PNG_DIR}"

print(f"Found {len(depth_png_files)} predicted depth PNGs")

for path in depth_png_files:
    img = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)

    if img is None:
        print(f"Warning: failed to read {path}")
        continue

    if img.ndim == 3:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    img = img.astype(np.float32)

    if img.max() > img.min():
        img_normalized = (img - img.min()) / (img.max() - img.min())
    else:
        img_normalized = img.copy()

    stem = path.stem
    np.save(RAW_NPY_DIR / f"{stem}_depth.npy", img_normalized)

print("Saved individual depth arrays to:", RAW_NPY_DIR)

Found 197 predicted depth PNGs
Saved individual depth arrays to: C:\Users\19734\Desktop\DAVIS\winter quarter 2026\EEC 174AY\Senior Design\Eco-Cars-Depth-Estimation-2026\data\depth_anything_v2\raw_npy


In [10]:
import numpy as np
from pathlib import Path

pred_npy_files = sorted(RAW_NPY_DIR.glob("*_depth.npy"))
assert len(pred_npy_files) > 0, f"No prediction npy files found in {RAW_NPY_DIR}"

pred_stack = []
pred_stems = []

for f in pred_npy_files:
    arr = np.load(f)
    pred_stack.append(arr)
    pred_stems.append(f.stem.replace("_depth", ""))

pred_stack = np.stack(pred_stack, axis=0)
pred_stems = np.array(pred_stems)

PRED_STACK_PATH = OUTPUT_DIR / "depthanythingv2_depths.npy"
PRED_STEMS_PATH = OUTPUT_DIR / "depthanythingv2_stems.npy"

np.save(PRED_STACK_PATH, pred_stack)
np.save(PRED_STEMS_PATH, pred_stems)

print("pred_stack shape:", pred_stack.shape)
print("Saved:", PRED_STACK_PATH)
print("Saved:", PRED_STEMS_PATH)

pred_stack shape: (197, 1280, 1920)
Saved: C:\Users\19734\Desktop\DAVIS\winter quarter 2026\EEC 174AY\Senior Design\Eco-Cars-Depth-Estimation-2026\data\depth_anything_v2\depthanythingv2_depths.npy
Saved: C:\Users\19734\Desktop\DAVIS\winter quarter 2026\EEC 174AY\Senior Design\Eco-Cars-Depth-Estimation-2026\data\depth_anything_v2\depthanythingv2_stems.npy


In [11]:
pred_stack = np.load(PRED_STACK_PATH, allow_pickle=True)
pred_stems = np.load(PRED_STEMS_PATH, allow_pickle=True)
gt_stack = np.load(GT_STACK_PATH, allow_pickle=True)
gt_stems = np.load(GT_STEMS_PATH, allow_pickle=True)

pred_dict = {stem: pred_stack[i] for i, stem in enumerate(pred_stems)}
gt_dict = {stem: gt_stack[i] for i, stem in enumerate(gt_stems)}

common_stems = sorted(set(pred_dict.keys()) & set(gt_dict.keys()))
assert len(common_stems) > 0, "No matching stems between prediction and ground truth"

pred_aligned = np.stack([pred_dict[s] for s in common_stems], axis=0)
gt_aligned = np.stack([gt_dict[s] for s in common_stems], axis=0)

print("Matched stems:", len(common_stems))
print("pred_aligned shape:", pred_aligned.shape)
print("gt_aligned shape:", gt_aligned.shape)
print("first 10 matched stems:", common_stems[:10])

Matched stems: 197
pred_aligned shape: (197, 1280, 1920)
gt_aligned shape: (197, 1280, 1920)
first 10 matched stems: ['frame_00000', 'frame_00001', 'frame_00002', 'frame_00003', 'frame_00004', 'frame_00005', 'frame_00006', 'frame_00007', 'frame_00008', 'frame_00009']


In [18]:
import numpy as np

def apply_custom_formula(pred_depth, formula_type=None, params=None):
    
    if formula_type is None or params is None:
        # 暂时没有公式时，先原样返回
        return pred_depth.copy()

    if formula_type == "linear":
        a = params["a"]
        b = params["b"]
        return a * pred_depth + b

    elif formula_type == "power":
        a = params["a"]
        b = params["b"]
        c = params.get("c", 0.0)
        return a * np.power(np.clip(pred_depth, 1e-6, None), b) + c

    elif formula_type == "exponential":
        a = params["a"]
        b = params["b"]
        c = params.get("c", 0.0)
        return a * np.exp(b * pred_depth) + c

    else:
        raise ValueError(f"Unknown formula_type: {formula_type}")

In [19]:
FORMULA_TYPE = None
FORMULA_PARAMS = None

pred_calibrated = np.stack(
    [apply_custom_formula(frame, FORMULA_TYPE, FORMULA_PARAMS) for frame in pred_aligned],
    axis=0
)

print("pred_calibrated shape:", pred_calibrated.shape)
print("Currently using placeholder calibration (identity mapping).")

pred_calibrated shape: (197, 1280, 1920)
Currently using placeholder calibration (identity mapping).


In [20]:
CALIBRATED_PATH = OUTPUT_DIR / "depthanythingv2_calibrated_placeholder.npy"
PRED_STACK_PATH = OUTPUT_DIR / "depthanythingv2_depths.npy"
PRED_STEMS_PATH = OUTPUT_DIR / "depthanythingv2_stems.npy"
GT_ALIGNED_PATH = OUTPUT_DIR / "groundtruth_aligned.npy"

np.save(CALIBRATED_PATH, pred_calibrated)
np.save(GT_ALIGNED_PATH, gt_aligned)

print("Saved placeholder calibrated depth to:", CALIBRATED_PATH)
print("Saved aligned ground truth to:", GT_ALIGNED_PATH)

Saved placeholder calibrated depth to: C:\Users\19734\Desktop\DAVIS\winter quarter 2026\EEC 174AY\Senior Design\Eco-Cars-Depth-Estimation-2026\data\depth_anything_v2\depthanythingv2_calibrated_placeholder.npy
Saved aligned ground truth to: C:\Users\19734\Desktop\DAVIS\winter quarter 2026\EEC 174AY\Senior Design\Eco-Cars-Depth-Estimation-2026\data\depth_anything_v2\groundtruth_aligned.npy
